# SIP ML Tool: Quality Control Workflow

* Author: Klaudio Peshtani
* Updated: May 22, 2026

This notebook is the main user-facing entry point for the SIP machine-learning quality-control workflow. It loads raw SIP spectra, builds one feature record per spectrum, trains the CatBoost screening model on labeled Good/Bad spectra, applies the physics-aware rule layer, and writes the final Good, Review, and Bad outputs. The workflow is organized into a Training/Validation stage and a Prediction/Reporting stage so new spectra can be screened from saved training artifacts without retraining the model each time.

The detailed implementation is organized in the `sip_ml_tool/` Python modules so that the notebook stays readable. Because the workflow uses relative paths, the notebook should be run from a package folder that contains the required files and folders listed below.

Required package contents:
- `SIP_ML_Tool.ipynb`
- `sip_ml_tool/`
- `Raw SIP Data/HPE SIP Data/Good data`
- `Raw SIP Data/HPE SIP Data/Bad data`
- `Raw SIP Data/HPE SIP Data/Cautious data`
- `Raw SIP Data/HPE SIP Data/Unknown data`
- `Final Results/Validation/model_artifacts/` if running the prediction stage without retraining

Primary raw-data inputs:
- `Raw SIP Data/HPE SIP Data/Good data`
- `Raw SIP Data/HPE SIP Data/Bad data`
- `Raw SIP Data/HPE SIP Data/Cautious data`
- `Raw SIP Data/HPE SIP Data/Unknown data`

Primary outputs are written to `Final Results/`. The top-level files are intended for reader-facing review, while `Final Results/Validation/` contains detailed technical outputs used for traceability.


## 1. Setup helper functions

This section imports the shared helper functions used by the workflow. The helper code lives in `sip_ml_tool/kk_features.py` and includes SIP file parsing utilities, plotting helpers, log-frequency calculations, and Kramers-Kronig consistency calculations.

These functions are separated from the notebook because they are lower-level utilities rather than workflow steps. Keeping them in a module makes the notebook easier to read while preserving the same execution behavior.


In [ ]:
# Helper functions used by the main workflow.
# The detailed helper implementation is stored in sip_ml_tool/kk_features.py
# so this notebook remains the main workflow without carrying all helper code inline.
from sip_ml_tool.kk_features import *


## 2. Load the core workflow modules

This section loads the main quality-control workflow from ordered Python modules. The module order matters because the workflow was developed as a sequence of feature, model, validation, and physics-rule updates. Later modules intentionally replace or extend earlier functions such as `build_feature_table`, `train_catboost`, `apply_override`, and `main()`.

The module layout is:
- `config.py`: imports, paths, thresholds, plotting settings, and output folders.
- `core_helpers.py`: filename parsing, frequency-band helpers, peak-picking, and basic rule-flag utilities.
- `data_io.py`: CSV discovery, HPE/field data loading, discovery audit generation, and raw-data metadata.
- `feature_table.py`: initial one-record-per-spectrum feature table construction.
- `catboost_model.py`: CatBoost model training and probability prediction helpers.
- `physics_rules.py`: initial physics-aware Good/Review/Bad rule layer.
- `plots_galleries.py`: spectrum plots, confusion/threshold figures, and PowerPoint gallery generation.
- `base_workflow.py`: original base workflow before later rule updates.
- `shape_qc.py`: additional shape metrics and rule refinements.
- `canonical_validation.py`: grouped cross-validation, threshold selection, and validation reporting.
- `rule_updates_v261_v266.py` and `rule_updates_v267_v271.py`: retained historical rule-update stack used by the current workflow.

Do not reorder the modules unless the rule-update stack is intentionally being refactored and revalidated.


In [ ]:
# Load the main workflow modules in order.
from sip_ml_tool.notebook_loader import load_notebook_module

CORE_WORKFLOW_MODULES = [
    "config.py",  # Imports, paths, constants, plotting theme, and output folders.
    "core_helpers.py",  # Small parsing, frequency-band, peak, and rule-flag helpers.
    "data_io.py",  # CSV readers, discovery audit, raw data loading, and UID metadata.
    "feature_table.py",  # Initial spectrum feature-table construction.
    "catboost_model.py",  # Initial CatBoost training and prediction helpers.
    "physics_rules.py",  # Initial physics-guided Good/Review/Bad override logic.
    "plots_galleries.py",  # Confusion plots, spectrum PNGs, and PowerPoint gallery creation.
    "base_workflow.py",  # Original base main workflow before later QC patches.
    "shape_qc.py",  # Shape/QC feature augmentation and intermediate override updates.
    "canonical_validation.py",  # V2.6 canonical grouped validation, thresholds, and model patch.
    "rule_updates_v261_v266.py",  # V2.6.1-V2.6.6 physics-rule update stack.
    "rule_updates_v267_v271.py",  # V2.6.7-V2.7.1 final base workflow update stack.
]

for _module_name in CORE_WORKFLOW_MODULES:
    load_notebook_module(_module_name, globals())


## 3. Apply the final rules and reporting layer

This section loads `sip_ml_tool/final_rules.py`, which applies the final grouped-validation settings, final triage policy, reader-facing report generation, and output-folder organization.

The final layer keeps the CatBoost model as a screening component, but the final Good/Review/Bad assignment is produced by the combined workflow. This is important because uncertain or physically inconsistent spectra can be routed to Review or Bad instead of being forced into a binary Good/Bad model decision.

After this cell runs, the notebook namespace contains the final `main()` function that should be executed in the last cell.


In [ ]:
# Final rules and reporting are kept in a Python module to shorten this notebook.
# They are executed in the notebook namespace because this section intentionally overrides
# functions defined in the main workflow modules above.
from pathlib import Path

_final_rules_path = Path("sip_ml_tool") / "final_rules.py"
exec(_final_rules_path.read_text(encoding="utf-8"), globals())


## 4. Run the staged workflow

Run this cell after the setup, core workflow modules, and final rules have loaded. The script prints numbered progress messages, elapsed times, and summary counts while it runs so users can see what is being loaded, trained, predicted, and exported. The first command runs the Training/Validation stage: it reads the raw data, builds the feature table, performs grouped validation, trains the CatBoost model, selects the physics-aware triage policy, and saves the model artifacts. The second command runs the Prediction/Reporting stage: it reloads those artifacts, predicts all spectra, applies the physics-aware triage layer, and writes the final outputs.

For a full fresh run, run the entire cell. For a faster prediction-only run after training artifacts already exist, run only `run_prediction_stage(...)`.

Expected high-level outputs:
- `Final Results/01_README.md`: guide to the output folder.
- `Final Results/02_QC_Summary_Report.md`: concise narrative summary of the run.
- `Final Results/03_QC_Proof_Scorecard.csv`: main validation scorecard comparing CatBoost-only and physics-aware triage.
- `Final Results/04_Final_Class_Counts.csv`: final Good/Review/Bad counts.
- `Final Results/05_Final_Decision_Queue.csv` and `06_Manual_Review_Queue.csv`: operational review tables.
- `Final Results/07_Gallery_GOOD.pptx`, `08_Gallery_REVIEW.pptx`, and `09_Gallery_BAD.pptx`: spectrum review galleries.
- `Final Results/Validation/`: detailed feature tables, predictions, grouped validation outputs, discovery audit, and supporting assets.


In [ ]:
# Stage 1: train/validate and save the model, thresholds, and triage-policy artifacts.
training_results = run_training_stage()

# Stage 2: load the saved artifacts, predict all spectra, apply physics-guided screening,
# and write the reader-facing outputs. For prediction-only reruns, run this stage by itself.
prediction_results = run_prediction_stage(
    use_cached_features=True,
    build_galleries=FINAL_BUILD_GALLERIES,
)
